In [1]:
import wurst
import wurst.searching as ws
import bw2io as bi
import bw2data as bd
import matplotlib.pyplot as plt

In [2]:
from prettytable import PrettyTable
from pprint import pprint
from collections import defaultdict

## 1. Seleccionamos nuestro proyecto

In [3]:
bd.projects.set_current("mobility") #Creating/accessing your project.

In [4]:
bd.databases

Databases dictionary with 9 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ei310-modified-transport
	ei_cutoff_3.10_remind_SSP2-Base_2020 2025-02-04
	ei_cutoff_3.10_remind_SSP2-Base_2050 2025-02-04
	ei_cutoff_3.10_remind_SSP2-PkBudg500_2020 2025-02-04
	ei_cutoff_3.10_remind_SSP2-PkBudg500_2050 2025-02-04
	parametric_LCA
	parametric_LCA_v1

## 2. Extraemos ecoinvent como un diccionario

Esto nos ayuda a filtrar, seleccionar, y modificar actividades de manera eficiente

In [5]:
db = wurst.extract_brightway2_databases("ecoinvent-3.10-cutoff")

Getting activity data


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23523/23523 [00:00<00:00, 82456.73it/s]


Adding exchange data to activities


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 743409/743409 [00:36<00:00, 20168.97it/s]


Filling out exchange data


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23523/23523 [00:02<00:00, 10538.40it/s]


In [6]:
print(f"db es ahora una: {type(db)}")
print(f"cada elemento de la lista es un: {type(db[0])}")

db es ahora una: <class 'list'>
cada elemento de la lista es un: <class 'dict'>


Veamos como es la estructura de cada actividad:

In [15]:
# Print a sample activity to see the structure
sample_activity = db[5475]  # Access the first activity
pprint(sample_activity)

{'categories': None,
 'classifications': [('ISIC rev.4 ecoinvent',
                      '3821:Treatment and disposal of non-hazardous waste'),
                     ('CPC', '39920: Sewage sludge')],
 'code': '3c9234071bcc8f3f1b16203de7f456af',
 'comment': 'The process represents the treatment of a waste (sewage sludge) '
            'that is exclusively generated from another waste treatment. '
            'Sewage sludge is generated in the wastewater treatment (WWT) of '
            "'wastewater from wafer fabrication'. Recommended use of this "
            'dataset: to be used within the specific chain of treatments for '
            "'wastewater from wafer fabrication'. Share of biogenic carbon in "
            'the sludge: 0%. Landfill gas capture and utilisation generates '
            '0.03321 kWh of electricity and 0.0599 MJ of useful heat. 0.002598 '
            'kg waste mass remains in the landfill even after the long term '
            'period.\n'
            'Geography:  La

## 3. Explorando las clasificaciones...

Vemos que 'classifications' pueder ser util.
1. Que 'classifications' existen en ecoinvent?
2. Que codigos existen para cada 'classification'?

In [8]:
# dictionary to store unique codes per classification system
classification_dict = defaultdict(set)
# dictionary to count occurrences of each classification system
classification_count = defaultdict(int)
# total number of activities
total_activities = len(db)

# iterate through all activities and extract classification systems & codes
for activity in db:
    classifications = activity.get("classifications", [])
    
    for class_type, class_code in classifications:
        classification_dict[class_type].add(class_code)
        classification_count[class_type] += 1 

# Print all unique classification systems with data coverage
print("Classification systems in ecoinvent:")
for class_type in sorted(classification_dict.keys()):  # Sorting for readability
    coverage = (classification_count[class_type] / total_activities) * 100  # Calculate percentage
    print(f" - {class_type} ({coverage:.2f}%)")
print("-" * 40)  # Separator

# Print classification systems and their unique codes
print("\nUnique Classification Systems and Their Codes:\n")
for class_type, unique_codes in classification_dict.items():
    print(f"{class_type} ({classification_count[class_type]}/{total_activities}, {classification_count[class_type] / total_activities * 100:.2f}% coverage):")
    for code in sorted(unique_codes):
        print(f" - {code}")
    print("-" * 40)

Classification systems in ecoinvent:
 - CPC (100.00%)
 - EcoSpold01Categories (36.70%)
 - ISIC rev.4 ecoinvent (98.88%)
----------------------------------------

Unique Classification Systems and Their Codes:

EcoSpold01Categories (8632/23523, 36.70% coverage):
 - agricultural means of production/buildings
 - agricultural means of production/feed
 - agricultural means of production/machinery
 - agricultural means of production/mineral fertiliser
 - agricultural means of production/organic fertiliser
 - agricultural means of production/pesticides
 - agricultural means of production/seed
 - agricultural means of production/work processes
 - agricultural production/animal production
 - agricultural production/plant production
 - biomass/cogeneration
 - biomass/fuels
 - biomass/production
 - building components/cladding
 - building components/doors
 - building components/windows
 - chemicals/inorganics
 - chemicals/organics
 - construction materials/additives
 - construction materials/bind

## 4. Ejemplo practico

1. Quiero encontrar todas las actividades que consumen 'transport' (en la unidad de 'ton km'), filtrando las que contienen en su nombre 'market for transport". Quiero filtrar los 'markets for transport' porque siempre consumen transporte y no quiero modificarlos. Pero quiero encontrar las actividades que consumen 'market for transport'.
2. Si la categoría CPC de esta actividad es '22130: Wheat', quiero aumentar cada 'transport' 'amount' en un 1%.

In [9]:
activity_filters = [
    ws.exclude(ws.contains("name", "market for transport"))
]

transport_filters = [
    ws.equals("unit", "ton kilometer"),
    ws.equals("type", "technosphere")
]

In [10]:
modifications = []

for ds in ws.get_many(db, *activity_filters):
    transport_exchanges = list(ws.technosphere(ds, *transport_filters))
    
    if not transport_exchanges:  # If no transport exchanges, skip
        continue

    classifications = ds.get("classifications", [])
    cpc_category = next((code for class_type, code in classifications if class_type == "CPC"), None)

    if cpc_category == "22130: Whey":
        for techno in transport_exchanges:
            old_amount = techno["amount"]
            new_amount = old_amount * 1.01 
            techno["amount"] = new_amount

            modifications.append([
                ds["name"],  
                cpc_category,  
                techno["name"],  
                round(old_amount, 6),  
                round(new_amount, 6)  
            ])


In [11]:
table = PrettyTable(["Activity Name", "CPC Category", "Transport Activity", "Old Amount", "New Amount"])
table.add_rows(modifications)
print(table)

+-----------------+--------------+---------------------------------------------------------+------------+------------+
|  Activity Name  | CPC Category |                    Transport Activity                   | Old Amount | New Amount |
+-----------------+--------------+---------------------------------------------------------+------------+------------+
| market for whey | 22130: Whey  |        market group for transport, freight train        |   0.0112   |  0.011312  |
| market for whey | 22130: Whey  | market group for transport, freight, lorry, unspecified |  0.01932   |  0.019513  |
+-----------------+--------------+---------------------------------------------------------+------------+------------+


In [13]:
wurst.write_brightway2_database(db, "ei310-modified-transport")

AssertionError: This database already exists